In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd

from core import (
    compare_benchmark,
    compare_table,
    db_connect,
    global_style,
    global_style_loose,
    grid_tuning,
    load_data,
)
from monte_carlo import propfirm_sim_leverage
from overfit_check import walk_forward_test
from params_finder import neighborhood_analysis, radius_diagnostics
from strats.dema import f0_be_exit
from visualizer import (
    combinations_compare_visualizer,
    portfolio_perf_visualizer,
    sensitivity_check_visualizer,
    trades_visualizer,
)

sys.path.insert(0, str(Path.cwd().parent))
import plotly.io as pio

# Get the absolute path to the 'src' directory
src_path = Path.cwd().parent
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))


pio.renderers.default = "notebook"

COMMISSION = 0.00015
CASH = 50_000
RISK_PER_TRADE = 10
RISK_PCT = RISK_PER_TRADE / CASH 

strat = f0_be_exit.STRAT(
   CASH=CASH,
   COMMISSION=COMMISSION,
   RISK_PER_TRADE=RISK_PER_TRADE,
)

# CONFIG
ASSET = "US100_scaled"
TIMEFRAME = "M5"
NAME = f"{ASSET}_{TIMEFRAME}"
TABLE_NAME = f'{strat.CODE}_{NAME}'
TABLE_NAME_WF = f"{TABLE_NAME}_WF"

DATA_PATH = f'~/market_data/{ASSET}/{NAME}.csv'
TARGET = 'profit_factor'

conn, cursor = db_connect()
ohlcv_data = load_data(DATA_PATH)

# params derived
PARAMS = strat.PARAMS
PARAM_COLS = list(PARAMS)
PARAMS_GRID = list(product(*(v["data"] for v in PARAMS.values())))
TOP_N = max(100, int(len(PARAMS_GRID) / 10))

# Print configuration constants for verification
config_df = pd.DataFrame.from_dict(
  {
    "ASSET": ASSET,
    "TABLE_NAME": TABLE_NAME,
    "STRAT": strat.NAME,
    "TARGET": TARGET,
    "RISK_PER_TRADE": RISK_PER_TRADE,
    "CASH": CASH,
    "COMMISSION": COMMISSION,
}
  , orient='index', columns=['Value'])

global_style_loose(config_df)

## Grid tuning

In [ ]:
# grid_tuning(params=PARAMS, table_name=TABLE_NAME, data=ohlcv_data, run_backtest=strat.run_backtest, force=True)
grid_tuning(params=PARAMS, table_name=TABLE_NAME, data=ohlcv_data, run_backtest=strat.run_backtest)

## Combinations compare for TOP vs All combinations

In [ ]:
def get_query(limit = True):
    return f"""
        SELECT *
        FROM {TABLE_NAME}
        ORDER BY {TARGET} DESC
        {f'LIMIT {TOP_N}' if limit else ''}
    """

top_df = pd.read_sql_query(get_query(), conn)
all_df = pd.read_sql_query(get_query(limit = False), conn)

combinations_compare_visualizer(
  first_data=top_df,
  second_data=all_df,
  chart_title=f"TOP {TOP_N} vs All combinations",
  first_name=f"TOP {TOP_N}",
  second_name="All combinations"
)

## Parameter neighborhood analysis 

In [ ]:
df = pd.read_sql_query(f"""
    SELECT *
    FROM {TABLE_NAME}
""", conn)

report, best_r = radius_diagnostics(df, PARAMS, TARGET, radii=(1, 2, 3, 4))
best_combination = neighborhood_analysis(df, PARAMS, TARGET=TARGET, radius=best_r)

## Sensitivity check

In [ ]:
df = pd.read_sql_query(f"""
    SELECT *
    FROM {TABLE_NAME}
""", conn)

sensitivity_check_visualizer(df, PARAM_COLS=PARAM_COLS, TARGET=TARGET)

## Benchmark

In [ ]:
best_combination_param = best_combination[PARAM_COLS].to_dict()

portfolio, signals = strat.run_backtest(ohlcv_data, params=best_combination_param)

compare_benchmark(
  data=ohlcv_data,
  portfolio=portfolio 
)

## Trade and Portfolio performance

In [ ]:
trades_visualizer(data=ohlcv_data, signals=signals, portfolio=portfolio, N = 500)
portfolio_perf_visualizer(portfolio)

## Calc the EV for propfirm trading (FTMO)

In [ ]:
port_value = portfolio.value()
log_returns = np.log(port_value / port_value.shift(1)).dropna()

# Estimate bars per day from log returns frequency (average count per calendar day)
bars_per_day = int(log_returns.groupby(log_returns.index.date).size().mean())

CAPITAL = 10_000
params = {
    "tp" : 1500 / CAPITAL,
    "sl" : 1000 / CAPITAL,
    "daily_sl" : 0.05,
    "trailing_sl" : False,
    "bars_per_day" : bars_per_day,
    "days_per_month" : 21,
    "n_days" : 252,
    "capital" : CAPITAL,
    "challenge_cost" : 100,
    "fixed_cost_month" : 1,
    "payout_pct" : 0.80,
    "bankroll" : 2_000,
}

params["bars_per_month"] = params["days_per_month"] * params["bars_per_day"]
params["n_bars"] = params["n_days"] * params["bars_per_day"]

leverage_list = range(1, 11)

df = propfirm_sim_leverage(
  log_returns=log_returns,
  leverage_list=leverage_list,
  params=params,
)
global_style(df)

## Calc the EV for propfirm trading (TOPSTEPX)

In [ ]:
port_value = portfolio.value()
log_returns = np.log(port_value / port_value.shift(1)).dropna()

bars_per_day = int(log_returns.groupby(log_returns.index.date).size().mean())

CAPITAL = 50_000
params = {
    "tp" : 3000 / CAPITAL,
    "sl" : 2000 / CAPITAL,
    "daily_sl" : 3000 / CAPITAL,
    "trailing_sl" : True,
    "bars_per_day" : bars_per_day,
    "days_per_month" : 21,
    "n_days" : 252,
    "capital" : CAPITAL,
    "challenge_cost" : 50,
    "fixed_cost_month" : 50,
    "payout_pct" : 0.90,
    "bankroll" : 2_000,
}

params["bars_per_month"] = params["days_per_month"] * params["bars_per_day"]
params["n_bars"] = params["n_days"] * params["bars_per_day"]

leverage_list = range(1, 11)

df = propfirm_sim_leverage(
  log_returns=log_returns,
  leverage_list=leverage_list,
  params=params,
)

global_style(df)

In [ ]:
raise SystemExit("Stopping execution at this cell.")

## Walk-forward test

In [ ]:
results, wfe = walk_forward_test(
  data=ohlcv_data,
  run_backtest=strat.run_backtest,
  table_name=TABLE_NAME_WF,
  param_grid={key: value["data"] for key, value in PARAMS.items()},
  is_years=2,
  oos_months=6,
  objective='profit_factor',
)